# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
import os
import chromadb
from dotenv import load_dotenv
from lib.agents import Agent
from lib.llm import LLM
from lib.tooling import tool
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.evaluation import TestCase, AgentEvaluator, EvaluationResult
from typing import List
from lib.messages import BaseMessage
from tavily import TavilyClient

In [3]:
# TODO: Load environment variables
load_dotenv()

True

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("agentic_ai_udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game
#
#    Tool semantic: def tool(func=None, *, name: str = None, description: str = None):
@tool(
        name="retrieve_game",
        description=f"""
            Semantic search: Finds most results in the vector DB.
            Args: query: a question about game industry.
            You'll receive results as list.
            Each element contains: Platform: like Game Boy, Playstation 5, Xbox 360, etc.
            Name: Name of the Game.
            YearOfRelease: Year when that game was released for that platform.
            Description: Additional details about the game.
        """
)
def retrieve_game(query: str) -> list:
    results = collection.query(
        query_texts=[query],
        n_results=5
    )
    return {"input": query, "output": results['metadatas'][0]}

#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
@tool(name="evaluate_retrieval", description=f"""
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question.
    Args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
""")
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    prompt = f"""
    Evaluate whether the documents are sufficient to answer the question.
    Start your response with exactly YES if they are sufficient or NO if they are not.
    Then give a brief explanation.
    
    Question: {question}
    
    Retrieved Documents: {retrieved_docs}
    """
    evaluation_llm = LLM(model="gpt-4o-mini", temperature=0.0)
    evaluation_result = evaluation_llm.invoke(input=prompt)
    
    # Here you would parse the evaluation_result into the useful and description fields
    # For example, you might want to use a specific format in your prompt to make parsing easier
    evaluation_text = evaluation_result.content or ""
    return {
        "input": {"question": question, "retrieved_docs": retrieved_docs},
        "output": {
            "useful": evaluation_text.strip().upper().startswith("YES"),
            "description": evaluation_text,
        }
    }

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry.
#    You'll receive results as list. Each element contains:
#    - title: title of the web page
#    - url: url of the web page
@tool(name="game_web_search", description=f"""
    Web search tool: Finds relevant web pages about the game industry.
    Args:
    - question: a question about game industry.
    You'll receive results as list. Each element contains:
    - title: title of the web page
    - url: url of the web page
""")
def game_web_search(question: str) -> list:
    # Here you would implement the logic to search the web using Tavily client
    # and return the results in the specified format
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY environment variable is not set.")
    client = TavilyClient(api_key=api_key)
    response = client.search(question)
    return {"input": question, "output": response['results']}

### Agent

In [7]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

instructions = """
You are a video game research agent.
For every user question, you MUST call retrieve_game before answering, even if the answer appears in the conversation history.
After retrieval, you MUST call evaluate_retrieval with the original question and retrieved documents.
If the evaluation says the documents are insufficient, call game_web_search before answering.
Do not answer from memory alone. Answer clearly and include source URLs when web search is used.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

In [8]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
test_questions = [
    ("question-1", "When Pokémon Gold and Silver was released?"),
    ("question-2", "Which one was the first 3D platformer Mario game?"),
    ("question-3", "Was Mortal Kombat X realeased for Playstation 5?"),
]

agent.reset_session()
runs = []
for session_id, question in test_questions:
    runs.append(agent.invoke(question, session_id=session_id))

run1, run2, run3 = runs

def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

print("\nMessages from run 1:")
messages1 = run1.get_final_state()["messages"]
print_messages(messages1)

print("\nMessages from run 2:")
messages2 = run2.get_final_state()["messages"]
print_messages(messages2)

print("\nMessages from run 3:")
messages3 = run3.get_final_state()["messages"]
print_messages(messages3)

# Demonstrate short-term memory with two related turns in the same session.
conversation_session_id = "conversation-memory-demo"
if conversation_session_id in agent.memory.get_all_sessions():
    agent.reset_session(conversation_session_id)

conversation_questions = [
    "When were Pokemon Gold and Silver released?",
    "Which platform were those two games released for?",
]
conversation_runs = [
    agent.invoke(question, session_id=conversation_session_id)
    for question in conversation_questions
]
conversation_messages = conversation_runs[-1].get_final_state()["messages"]
remembered_user_turns = [
    message.content for message in conversation_messages if message.role == "user"
]
follow_up_tools = [
    call.function.name
    for message in conversation_messages
    for call in (getattr(message, "tool_calls", None) or [])
]
follow_up_answer = next(
    message.content
    for message in reversed(conversation_messages)
    if message.role == "assistant" and message.content
)

assert len(agent.get_session_runs(conversation_session_id)) == 2
assert remembered_user_turns[-2:] == conversation_questions

print("\nSame-session memory demonstration:")
print(f"Session ID: {conversation_session_id}")
print(f"Stored runs: {len(agent.get_session_runs(conversation_session_id))}")
print(f"Remembered user turns: {remembered_user_turns[-2:]}")
print(f"Tools visible in accumulated history: {' -> '.join(follow_up_tools)}")
print(f"Follow-up answer:\n{follow_up_answer}")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


...

[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Same-session memory demonstration:
Session ID: conversation-memory-demo
Stored runs: 2
Remembered user turns: ['When were Pokemon Gold and Silver released?', 'Which platform were those two games released for?']
Tools visible in accumulated history: retrieve_game -> evaluate_retrieval -> retrieve_game -> evaluate_retrieval
Follow-up answer:
"Pokémon Gold and Silver" were released for the Game Boy Color.


### (Optional) Advanced

In [9]:
# TODO: Update your agent with long-term memory
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager

MEMORY_OWNER = "udaplay-user"
MEMORY_NAMESPACE = "game-research"

# LongTermMemory creates its collection with force=True, so keep the same
# instance when this cell is re-run to avoid erasing stored memories.
if "long_term_memory" not in globals():
    openai_api_key = os.getenv("OPENAI_API_KEY")
    if not openai_api_key:
        raise ValueError("OPENAI_API_KEY environment variable is not set.")

    memory_db = VectorStoreManager(openai_api_key=openai_api_key)
    long_term_memory = LongTermMemory(memory_db)

@tool(
    name="recall_game_memory",
    description="""
        Search long-term memory for facts learned during earlier research sessions.
        Args: query: the current game-related question or search phrase.
        Returns semantically related memories with their timestamps and distances.
    """,
)
def recall_game_memory(query: str) -> dict:
    result = long_term_memory.search(
        query_text=query,
        owner=MEMORY_OWNER,
        namespace=MEMORY_NAMESPACE,
        limit=3,
    )
    memories = [
        {
            "content": fragment.content,
            "timestamp": fragment.timestamp,
            "distance": distance,
        }
        for fragment, distance in zip(
            result.fragments,
            result.metadata.get("distances", []),
        )
    ]
    return {"input": query, "output": memories}

@tool(
    name="store_game_memory",
    description="""
        Store one concise, verified game fact for use in future sessions.
        Args: fact: the verified fact; source: optional supporting source URL.
    """,
)
def store_game_memory(fact: str, source: str = "") -> dict:
    if source and source not in fact:
        fact = f"{fact} Source: {source}"
    fragment = MemoryFragment(
        content=fact,
        owner=MEMORY_OWNER,
        namespace=MEMORY_NAMESPACE,
    )
    long_term_memory.register(
        fragment,
        metadata={"kind": "verified-game-fact"},
    )
    return {
        "input": fact,
        "output": {"stored": True, "timestamp": fragment.timestamp},
    }

long_term_instructions = """
You are a video game research agent with long-term memory.
For every user question, perform each workflow step exactly once unless a tool fails:
1. Call recall_game_memory to retrieve relevant facts from earlier sessions.
2. Call retrieve_game to search the local game database.
3. Call evaluate_retrieval with the original question and retrieved documents.
4. If the retrieved documents are insufficient, call game_web_search.
5. Before the final answer, call store_game_memory with one concise, verified fact
   containing the answer and any relevant source URL.
Long-term memory is supporting context, not a substitute for verification.
Answer clearly and include source URLs when web search is used.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=long_term_instructions,
    tools=[
        recall_game_memory,
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
        store_game_memory,
    ],
    temperature=0.0,
)

print("Long-term-memory agent ready.")


Long-term-memory agent ready.


In [10]:
# TODO: Test your agent with long-term memory by invoking it with the same questions again.
import ast
import json
import io
from contextlib import redirect_stdout

def get_final_answer(run) -> str:
    final_state = run.get_final_state() or {}
    for message in reversed(final_state.get("messages", [])):
        if message.role == "assistant" and message.content:
            return message.content
    return "No final assistant answer was produced."

def get_called_tools(run) -> List[str]:
    final_state = run.get_final_state() or {}
    tool_names = []
    for message in final_state.get("messages", []):
        for call in getattr(message, "tool_calls", None) or []:
            tool_names.append(call.function.name)
    return tool_names

def get_recalled_memory_count(run) -> int:
    final_state = run.get_final_state() or {}
    for message in final_state.get("messages", []):
        if message.role == "tool" and getattr(message, "name", None) == "recall_game_memory":
            try:
                payload = ast.literal_eval(json.loads(message.content))
                return len(payload.get("output", []))
            except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                return 0
    return 0

agent.reset_session()
long_term_runs = []

# Capture verbose StateMachine tracing and print only useful results below.
with redirect_stdout(io.StringIO()):
    for index, (_, question) in enumerate(test_questions, start=1):
        long_term_runs.append(
            agent.invoke(question, session_id=f"long-term-question-{index}")
        )

long_term_answers = [get_final_answer(run) for run in long_term_runs]
long_term_final_states = [run.get_final_state() for run in long_term_runs]

for index, ((_, question), run, answer) in enumerate(
    zip(test_questions, long_term_runs, long_term_answers),
    start=1,
):
    print(f"\n{'=' * 80}")
    print(f"Question {index}: {question}")
    print(f"\nLong-term-memory agent answer:\n{answer}")
    print(f"\nTools called: {' -> '.join(get_called_tools(run)) or 'none'}")
    print(f"Memories recalled: {get_recalled_memory_count(run)}")



Question 1: When Pokémon Gold and Silver was released?

Long-term-memory agent answer:
Pokémon Gold and Silver was released in 1999 for the Game Boy Color.

Tools called: recall_game_memory -> retrieve_game -> evaluate_retrieval -> store_game_memory
Memories recalled: 0

Question 2: Which one was the first 3D platformer Mario game?

Long-term-memory agent answer:
The first 3D platformer Mario game is **Super Mario 64**, which was released in **1996** for the **Nintendo 64**. This game set new standards for the genre and featured Mario's quest to rescue Princess Peach.

Tools called: recall_game_memory -> retrieve_game -> evaluate_retrieval -> store_game_memory
Memories recalled: 1

Question 3: Was Mortal Kombat X realeased for Playstation 5?

Long-term-memory agent answer:
Mortal Kombat X was originally released for PlayStation 4, but it is playable on PlayStation 5. Some features available on PS4 may be absent when played on PS5. 

For more details, you can visit the official PlaySta

In [11]:
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes
import json
from typing import Any, Dict, List, TypedDict

class GameResearchState(TypedDict, total=False):
    question: str
    memories: List[Dict[str, Any]]
    retrieved_docs: List[Dict[str, Any]]
    evaluation: Dict[str, Any]
    web_results: List[Dict[str, Any]]
    answer: str
    memory_saved: bool

# Step requires regular functions that accept the state dictionary. These
# adapters make each previously defined Tool an explicit workflow node.
def recall_memory_node(state: GameResearchState) -> dict:
    result = recall_game_memory(query=state["question"])
    return {"memories": result["output"]}

def retrieve_game_node(state: GameResearchState) -> dict:
    result = retrieve_game(query=state["question"])
    return {"retrieved_docs": result["output"]}

def evaluate_retrieval_node(state: GameResearchState) -> dict:
    result = evaluate_retrieval(
        question=state["question"],
        retrieved_docs=state.get("retrieved_docs", []),
    )
    return {"evaluation": result["output"]}

def route_after_evaluation(state: GameResearchState) -> str:
    if state.get("evaluation", {}).get("useful", False):
        return "synthesize_answer"
    return "game_web_search"

def game_web_search_node(state: GameResearchState) -> dict:
    result = game_web_search(question=state["question"])
    return {"web_results": result["output"]}

answer_llm = LLM(model="gpt-4o-mini", temperature=0.0)

def synthesize_answer_node(state: GameResearchState) -> dict:
    evidence = {
        "long_term_memories": state.get("memories", []),
        "local_documents": state.get("retrieved_docs", []),
        "retrieval_evaluation": state.get("evaluation", {}),
        "web_results": state.get("web_results", []),
    }
    prompt = f"""
    Answer the video game question using only the supplied evidence.
    Prefer local documents when they are sufficient. If web results were used,
    include the relevant source URL. Be concise and do not mention this workflow.

    Question: {state['question']}
    Evidence: {json.dumps(evidence, default=str, indent=2)}
    """
    response = answer_llm.invoke(input=prompt)
    return {"answer": response.content or ""}

def store_memory_node(state: GameResearchState) -> dict:
    fact = f"Question: {state['question']} Answer: {state['answer']}"
    result = store_game_memory(fact=fact)
    return {"memory_saved": result["output"]["stored"]}

entry_node = EntryPoint[GameResearchState]()
recall_node = Step[GameResearchState]("recall_game_memory", recall_memory_node)
retrieve_node = Step[GameResearchState]("retrieve_game", retrieve_game_node)
evaluate_node = Step[GameResearchState]("evaluate_retrieval", evaluate_retrieval_node)
web_search_node = Step[GameResearchState]("game_web_search", game_web_search_node)
answer_node = Step[GameResearchState]("synthesize_answer", synthesize_answer_node)
store_node = Step[GameResearchState]("store_game_memory", store_memory_node)
termination_node = Termination[GameResearchState]()

game_research_machine = StateMachine[GameResearchState](GameResearchState)
game_research_machine.add_steps([
    entry_node,
    recall_node,
    retrieve_node,
    evaluate_node,
    web_search_node,
    answer_node,
    store_node,
    termination_node,
])

game_research_machine.connect(entry_node, recall_node)
game_research_machine.connect(recall_node, retrieve_node)
game_research_machine.connect(retrieve_node, evaluate_node)
game_research_machine.connect(
    evaluate_node,
    [answer_node, web_search_node],
    condition=route_after_evaluation,
)
game_research_machine.connect(web_search_node, answer_node)
game_research_machine.connect(answer_node, store_node)
game_research_machine.connect(store_node, termination_node)

def research_game(question: str):
    run = game_research_machine.run({"question": question})
    return run, run.get_final_state()

print("Explicit game-research state machine ready.")


Explicit game-research state machine ready.


In [12]:
# TODO: Test the state machine with the same questions again, and compare
# the messages and final states with the previous agent implementation.
previous_runs = [run1, run2, run3]
previous_answers = [get_final_answer(run) for run in previous_runs]

state_machine_runs = []
state_machine_final_states = []

with redirect_stdout(io.StringIO()):
    for _, question in test_questions:
        machine_run, machine_state = research_game(question)
        state_machine_runs.append(machine_run)
        state_machine_final_states.append(machine_state)

for index, (_, question) in enumerate(test_questions):
    previous_run = previous_runs[index]
    long_term_run = long_term_runs[index]
    machine_run = state_machine_runs[index]
    machine_state = state_machine_final_states[index]

    previous_message_count = len(previous_run.get_final_state().get("messages", []))
    long_term_message_count = len(long_term_run.get_final_state().get("messages", []))
    node_path = [snapshot.step_id for snapshot in machine_run.snapshots]

    compact_machine_state = {
        "answer": machine_state.get("answer"),
        "retrieval_useful": machine_state.get("evaluation", {}).get("useful"),
        "used_web_search": bool(machine_state.get("web_results")),
        "memory_saved": machine_state.get("memory_saved", False),
        "memories_recalled": len(machine_state.get("memories", [])),
        "documents_retrieved": len(machine_state.get("retrieved_docs", [])),
    }

    print(f"\n{'#' * 88}")
    print(f"Question {index + 1}: {question}")
    print(f"\nPrevious agent answer:\n{previous_answers[index]}")
    print(f"Previous agent messages: {previous_message_count}")
    print(f"Previous agent tools: {' -> '.join(get_called_tools(previous_run)) or 'none'}")

    print(f"\nLong-term-memory agent answer:\n{long_term_answers[index]}")
    print(f"Long-term-memory messages: {long_term_message_count}")
    print(f"Long-term-memory tools: {' -> '.join(get_called_tools(long_term_run)) or 'none'}")

    print(f"\nExplicit state-machine answer:\n{machine_state.get('answer', '')}")
    print(f"State-machine node path: {' -> '.join(node_path)}")
    print("State-machine final state summary:")
    print(json.dumps(compact_machine_state, indent=2, ensure_ascii=False))



########################################################################################
Question 1: When Pokémon Gold and Silver was released?

Previous agent answer:
Pokémon Gold and Silver was released in 1999 for the Game Boy Color.
Previous agent messages: 7
Previous agent tools: retrieve_game -> evaluate_retrieval

Long-term-memory agent answer:
Pokémon Gold and Silver was released in 1999 for the Game Boy Color.
Long-term-memory messages: 11
Long-term-memory tools: recall_game_memory -> retrieve_game -> evaluate_retrieval -> store_game_memory

Explicit state-machine answer:
Pokémon Gold and Silver was released in 1999 for the Game Boy Color.
State-machine node path: __entry__ -> recall_game_memory -> retrieve_game -> evaluate_retrieval -> synthesize_answer -> store_game_memory
State-machine final state summary:
{
  "answer": "Pokémon Gold and Silver was released in 1999 for the Game Boy Color.",
  "retrieval_useful": true,
  "used_web_search": false,
  "memory_saved": true,
  "

## Standout Enhancements

The following features are independent from the core project implementation above.

### Personalized Dataset

In [13]:
import json
from pathlib import Path

# Synchronize games/*.json into the existing persistent collection.
# Unchanged records are skipped, so rerunning this cell is inexpensive.
existing = collection.get(include=["metadatas"])
existing_games = dict(zip(existing["ids"], existing["metadatas"]))
updated_game_ids = []

for game_path in sorted(Path("games").glob("*.json")):
    with game_path.open("r", encoding="utf-8") as game_file:
        game = json.load(game_file)

    game_id = game_path.stem
    if existing_games.get(game_id) == game:
        continue

    document = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"{game['Description']}"
    )
    collection.upsert(
        ids=[game_id],
        documents=[document],
        metadatas=[game],
    )
    updated_game_ids.append(game_id)

print(f"Knowledge base contains {collection.count()} games.")
print(f"Updated records: {updated_game_ids or 'none'}")


Knowledge base contains 25 games.
Updated records: none


### Advanced Memory: Persistent Web Knowledge

In [14]:
import ast
import hashlib
import json
import os
import time
from chromadb.utils import embedding_functions

PERSISTENT_MEMORY_OWNER = "udaplay-user"
PERSISTENT_MEMORY_NAMESPACE = "verified-web-research"

persistent_memory_embedding = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
)
persistent_memory_collection = chroma_client.get_or_create_collection(
    name="udaplay_persistent_web_memory",
    embedding_function=persistent_memory_embedding,
)

def decode_tool_message(content: str) -> dict:
    """Decode the string-wrapped dictionaries produced by Agent._tool_step."""
    try:
        decoded = json.loads(content)
        return ast.literal_eval(decoded) if isinstance(decoded, str) else decoded
    except (json.JSONDecodeError, ValueError, SyntaxError, TypeError):
        return {}

@tool(
    name="recall_persistent_game_memory",
    description="Search disk-backed memory for verified facts learned from earlier web searches.",
)
def recall_persistent_game_memory(query: str) -> dict:
    if persistent_memory_collection.count() == 0:
        return {"input": query, "output": []}

    result = persistent_memory_collection.query(
        query_texts=[query],
        n_results=min(3, persistent_memory_collection.count()),
        where={
            "$and": [
                {"owner": {"$eq": PERSISTENT_MEMORY_OWNER}},
                {"namespace": {"$eq": PERSISTENT_MEMORY_NAMESPACE}},
            ]
        },
        include=["documents", "distances", "metadatas"],
    )
    memories = [
        {
            "fact": fact,
            "source_url": metadata.get("source_url", ""),
            "timestamp": metadata.get("timestamp"),
            "distance": distance,
        }
        for fact, metadata, distance in zip(
            result.get("documents", [[]])[0],
            result.get("metadatas", [[]])[0],
            result.get("distances", [[]])[0],
        )
    ]
    return {"input": query, "output": memories}

@tool(
    name="store_verified_web_fact",
    description="Store a concise verified game fact and its supporting web source URL on disk.",
)
def store_verified_web_fact(fact: str, source_url: str) -> dict:
    memory_key = f"{PERSISTENT_MEMORY_OWNER}|{source_url}|{fact}"
    memory_id = hashlib.sha256(memory_key.encode("utf-8")).hexdigest()
    metadata = {
        "owner": PERSISTENT_MEMORY_OWNER,
        "namespace": PERSISTENT_MEMORY_NAMESPACE,
        "source_url": source_url,
        "timestamp": int(time.time()),
        "learned_from_web": True,
    }
    persistent_memory_collection.upsert(
        ids=[memory_id],
        documents=[fact],
        metadatas=[metadata],
    )
    return {"input": fact, "output": {"stored": True, **metadata}}

persistent_memory_instructions = """
You are a game research agent with disk-backed memory.
First call recall_persistent_game_memory, then retrieve_game and evaluate_retrieval.
If local evidence is insufficient, call game_web_search. When web search is used,
call store_verified_web_fact with one verified fact and its source URL before answering.
Always verify remembered facts against local or web evidence.
"""

persistent_memory_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=persistent_memory_instructions,
    tools=[
        recall_persistent_game_memory,
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
        store_verified_web_fact,
    ],
    temperature=0.0,
)

def persistent_final_answer(run) -> str:
    messages = (run.get_final_state() or {}).get("messages", [])
    for message in reversed(messages):
        if message.role == "assistant" and message.content:
            return message.content
    return ""

def run_persistent_memory_agent(question: str, session_id: str = "persistent-demo"):
    persistent_memory_agent.reset_session()
    run = persistent_memory_agent.invoke(question, session_id=session_id)
    messages = (run.get_final_state() or {}).get("messages", [])
    called_tools = [
        call.function.name
        for message in messages
        for call in (getattr(message, "tool_calls", None) or [])
    ]

    # Guarantee learning even if the model forgets the final storage tool call.
    if "game_web_search" in called_tools and "store_verified_web_fact" not in called_tools:
        answer = persistent_final_answer(run)
        source_url = ""
        for message in messages:
            if message.role == "tool" and getattr(message, "name", None) == "game_web_search":
                web_payload = decode_tool_message(message.content)
                web_results = web_payload.get("output", [])
                source_url = web_results[0].get("url", "") if web_results else ""
                break
        if source_url:
            store_verified_web_fact(fact=answer, source_url=source_url)

    return run

print(f"Persistent web memories available: {persistent_memory_collection.count()}")


Persistent web memories available: 1


### Structured Output

In [15]:
import io
import re
from contextlib import redirect_stdout
from typing import List
from pydantic import BaseModel, Field

class StructuredGameAnswer(BaseModel):
    question: str
    answer: str
    sources: List[str] = Field(default_factory=list)
    used_web_search: bool
    retrieval_useful: bool
    memories_recalled: int
    documents_retrieved: int
    workflow: List[str]
    confidence: float = Field(ge=0.0, le=1.0)

SOURCE_URL_PATTERN = re.compile(r"https?://[^\s\]\)>\"']+")

def extract_source_urls(value: str) -> List[str]:
    if not isinstance(value, str):
        return []
    return [match.rstrip(".,;:") for match in SOURCE_URL_PATTERN.findall(value)]

def collect_source_urls(state: dict, answer: str) -> List[str]:
    candidates = extract_source_urls(answer)

    for result in state.get("web_results", []):
        if result.get("url"):
            candidates.append(result["url"])

    for memory in state.get("memories", []):
        if memory.get("source_url"):
            candidates.append(memory["source_url"])
        candidates.extend(extract_source_urls(memory.get("content", "")))
        candidates.extend(extract_source_urls(memory.get("fact", "")))

    for document in state.get("retrieved_docs", []):
        for key in ("url", "source_url", "SourceURL"):
            if document.get(key):
                candidates.append(document[key])

    # Preserve evidence order while removing duplicates.
    return list(dict.fromkeys(candidates))

def research_game_with_structured_output(question: str) -> dict:
    with redirect_stdout(io.StringIO()):
        run, state = research_game(question)

    web_results = state.get("web_results", [])
    answer = state.get("answer", "")
    sources = collect_source_urls(state, answer)
    retrieval_useful = state.get("evaluation", {}).get("useful", False)
    structured = StructuredGameAnswer(
        question=question,
        answer=answer,
        sources=sources,
        used_web_search=bool(web_results),
        retrieval_useful=retrieval_useful,
        memories_recalled=len(state.get("memories", [])),
        documents_retrieved=len(state.get("retrieved_docs", [])),
        workflow=[snapshot.step_id for snapshot in run.snapshots],
        confidence=0.9 if retrieval_useful else 0.75 if web_results else 0.5,
    )
    return {
        "natural_language": structured.answer,
        "structured_json": structured.model_dump(),
        "run": run,
        "final_state": state,
    }

def print_structured_game_answer(result: dict):
    print("Natural-language answer:")
    print(result["natural_language"])
    print("\nStructured JSON:")
    print(json.dumps(result["structured_json"], indent=2, ensure_ascii=False))

print("Structured-output wrapper ready: research_game_with_structured_output(question)")


Structured-output wrapper ready: research_game_with_structured_output(question)


### Visualization Dashboard

In [16]:
from collections import Counter
from html import escape
from IPython.display import HTML, display

def render_research_dashboard(run=None, final_state=None):
    records = collection.get(include=["metadatas"]).get("metadatas", [])
    platforms = Counter(record.get("Platform", "Unknown") for record in records)
    genres = Counter(record.get("Genre", "Unknown") for record in records)
    max_platform = max(platforms.values(), default=1)
    max_genre = max(genres.values(), default=1)

    def bar_rows(items, maximum):
        return "".join(
            f"<div style='margin:8px 0'>"
            f"<div style='display:flex;justify-content:space-between'>"
            f"<span>{escape(str(label))}</span><strong>{count}</strong></div>"
            f"<div style='height:8px;background:#e5e7eb;border-radius:99px'>"
            f"<div style='width:{100 * count / maximum:.0f}%;height:8px;"
            f"background:#6366f1;border-radius:99px'></div></div></div>"
            for label, count in items
        )

    node_path = [snapshot.step_id for snapshot in run.snapshots] if run else []
    state = final_state or {}
    workflow_html = " -> ".join(escape(node) for node in node_path) or "Run a query to visualize its workflow."
    metrics = {
        "Games": len(records),
        "Platforms": len(platforms),
        "Genres": len(genres),
        "Persistent memories": persistent_memory_collection.count(),
        "Retrieved docs": len(state.get("retrieved_docs", [])),
        "Memory hits": len(state.get("memories", [])),
    }
    metric_cards = "".join(
        f"<div style='background:white;padding:14px;border-radius:12px;"
        f"box-shadow:0 1px 3px #0002'><div style='color:#6b7280'>{escape(label)}</div>"
        f"<div style='font-size:26px;font-weight:700'>{value}</div></div>"
        for label, value in metrics.items()
    )

    dashboard = f"""
    <div style='font-family:Arial,sans-serif;background:#f8fafc;padding:20px;border-radius:16px'>
      <h2 style='margin-top:0'>UdaPlay Research Dashboard</h2>
      <div style='display:grid;grid-template-columns:repeat(3,minmax(120px,1fr));gap:12px'>
        {metric_cards}
      </div>
      <div style='display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-top:20px'>
        <div style='background:white;padding:16px;border-radius:12px'>
          <h3>Top Platforms</h3>{bar_rows(platforms.most_common(6), max_platform)}
        </div>
        <div style='background:white;padding:16px;border-radius:12px'>
          <h3>Top Genres</h3>{bar_rows(genres.most_common(6), max_genre)}
        </div>
      </div>
      <div style='background:#111827;color:#f9fafb;padding:16px;border-radius:12px;margin-top:20px'>
        <strong>Latest workflow</strong><div style='margin-top:8px'>{workflow_html}</div>
      </div>
    </div>
    """
    display(HTML(dashboard))

render_research_dashboard()


### Custom Tools

In [17]:
import json
import os
import re
from datetime import datetime

POSITIVE_REVIEW_WORDS = {
    "amazing", "beautiful", "excellent", "fun", "great", "immersive",
    "innovative", "masterpiece", "polished", "rewarding", "smooth",
}
NEGATIVE_REVIEW_WORDS = {
    "boring", "broken", "buggy", "confusing", "disappointing", "grindy",
    "repetitive", "slow", "tedious", "unfinished", "unfair", "weak",
}

@tool(
    name="analyze_game_review_sentiment",
    description="Analyze a game review and return transparent sentiment evidence.",
)
def analyze_game_review_sentiment(review: str) -> dict:
    tokens = re.findall(r"[a-zA-Z']+", review.lower())
    positive_hits = [word for word in tokens if word in POSITIVE_REVIEW_WORDS]
    negative_hits = [word for word in tokens if word in NEGATIVE_REVIEW_WORDS]
    score = len(positive_hits) - len(negative_hits)
    sentiment = "positive" if score > 0 else "negative" if score < 0 else "neutral"
    matched_terms = len(positive_hits) + len(negative_hits)
    confidence = round(abs(score) / matched_terms, 2) if matched_terms else 0.0
    return {
        "input": review,
        "output": {
            "sentiment": sentiment,
            "score": score,
            "confidence": confidence,
            "positive_terms": positive_hits,
            "negative_terms": negative_hits,
        },
    }

@tool(
    name="detect_trending_games",
    description="Search the web for currently trending games for a platform or topic.",
)
def detect_trending_games(topic: str = "video games", platform: str = "all") -> dict:
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY environment variable is not set.")
    query = f"currently trending {topic} games {platform} {datetime.now().year}"
    response = TavilyClient(api_key=api_key).search(query, max_results=5)
    results = [
        {
            "title": item.get("title"),
            "url": item.get("url"),
            "summary": item.get("content"),
            "score": item.get("score"),
        }
        for item in response.get("results", [])
    ]
    return {"input": {"topic": topic, "platform": platform}, "output": results}

custom_tools_agent = Agent(
    model_name="gpt-4o-mini",
    instructions="""
    You are a game-industry analyst. Use analyze_game_review_sentiment for review
    analysis and detect_trending_games for current trend questions. Cite URLs for trends.
    """,
    tools=[analyze_game_review_sentiment, detect_trending_games],
    temperature=0.0,
)

sentiment_demo = analyze_game_review_sentiment(
    "A beautiful and immersive game, but the late missions become repetitive."
)
print(json.dumps(sentiment_demo["output"], indent=2))


{
  "sentiment": "positive",
  "score": 1,
  "confidence": 0.33,
  "positive_terms": [
    "beautiful",
    "immersive"
  ],
  "negative_terms": [
    "repetitive"
  ]
}


In [18]:
# TEST 1: Persistent memory learns from web research and survives new sessions
def compact_tool_output(tool_name: str, payload: dict) -> dict:
    output = payload.get("output")
    if tool_name in {"recall_persistent_game_memory", "game_web_search"}:
        return {
            "result_count": len(output or []),
            "results": (output or [])[:3],
        }
    if tool_name == "retrieve_game":
        return {
            "result_count": len(output or []),
            "games": [item.get("Name") for item in (output or [])[:5]],
        }
    return output if isinstance(output, dict) else {"value": output}

def print_detailed_agent_run(label: str, question: str, run):
    final_state = run.get_final_state() or {}
    messages = final_state.get("messages", [])
    print(f"\n{'=' * 96}")
    print(label)
    print(f"Question: {question}")
    print(f"Workflow snapshots: {' -> '.join(snapshot.step_id for snapshot in run.snapshots)}")

    tool_index = 0
    for message in messages:
        for call in getattr(message, "tool_calls", None) or []:
            tool_index += 1
            try:
                arguments = json.loads(call.function.arguments)
            except json.JSONDecodeError:
                arguments = call.function.arguments
            print(f"\nTool call {tool_index}: {call.function.name}")
            print(f"Arguments: {json.dumps(arguments, indent=2, ensure_ascii=False) if isinstance(arguments, dict) else arguments}")

        if message.role == "tool":
            payload = decode_tool_message(message.content)
            summary = compact_tool_output(message.name, payload)
            print(f"Tool result ({message.name}):")
            print(json.dumps(summary, indent=2, ensure_ascii=False, default=str))

    print("\nFinal answer:")
    print(persistent_final_answer(run))
    print(f"Messages: {len(messages)} | Tool calls: {tool_index}")

persistent_test_question = "Was Mortal Kombat X released natively for PlayStation 5?"
memory_count_before = persistent_memory_collection.count()

with redirect_stdout(io.StringIO()):
    persistent_first_run = run_persistent_memory_agent(
        persistent_test_question,
        session_id="persistent-memory-first-pass",
    )

memory_count_after_first = persistent_memory_collection.count()
persistent_recall_result = recall_persistent_game_memory(persistent_test_question)

with redirect_stdout(io.StringIO()):
    persistent_second_run = run_persistent_memory_agent(
        persistent_test_question,
        session_id="persistent-memory-second-pass",
    )

print_detailed_agent_run(
    "Persistent-memory agent: first pass",
    persistent_test_question,
    persistent_first_run,
)
print_detailed_agent_run(
    "Persistent-memory agent: second pass (new short-term session)",
    persistent_test_question,
    persistent_second_run,
)
print("\nPersistent memory verification:")
print(json.dumps({
    "memory_count_before": memory_count_before,
    "memory_count_after_first_pass": memory_count_after_first,
    "recalled_memories": persistent_recall_result["output"],
}, indent=2, ensure_ascii=False, default=str))



Persistent-memory agent: first pass
Question: Was Mortal Kombat X released natively for PlayStation 5?
Workflow snapshots: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor

Tool call 1: recall_persistent_game_memory
Arguments: {
  "query": "Mortal Kombat X release platforms"
}
Tool result (recall_persistent_game_memory):
{
  "result_count": 1,
  "results": [
    {
      "fact": "Mortal Kombat X was not released natively for PlayStation 5 but is available through the PlayStation Plus Collection.",
      "source_url": "https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V",
      "timestamp": 1780860360,
      "distance": 0.16615909337997437
    }
  ]
}

Tool call 2: evaluate_retrieval
Arguments: {
  "question": "Was Mortal Kombat X released natively for PlayStation 5?",
  "retrieved_docs": "Mortal Kombat X was not released natively for PlayStation 5 but is available 

In [19]:
# TEST 2: Natural-language and structured JSON answers
structured_test_questions = [
    "Which game in the local knowledge base is a FromSoftware open-world action RPG?",
    "Was Mortal Kombat X released natively for PlayStation 5?",
]
structured_test_results = []

for question in structured_test_questions:
    result = research_game_with_structured_output(question)
    structured_test_results.append(result)
    print(f"\n{'=' * 96}")
    print(f"Question: {question}")
    print_structured_game_answer(result)
    cited_urls = extract_source_urls(result["natural_language"])
    missing_citations = [
        url for url in cited_urls if url not in result["structured_json"]["sources"]
    ]
    assert not missing_citations, f"Citations missing from structured sources: {missing_citations}"

    print("\nFinal-state evidence summary:")
    state = result["final_state"]
    print(json.dumps({
        "retrieved_games": [
            item.get("Name") for item in state.get("retrieved_docs", [])
        ],
        "retrieval_evaluation": state.get("evaluation", {}),
        "web_sources": [
            item.get("url") for item in state.get("web_results", [])
        ],
        "answer_citations": cited_urls,
        "citation_alignment": "PASS",
        "memory_saved": state.get("memory_saved", False),
    }, indent=2, ensure_ascii=False, default=str))



Question: Which game in the local knowledge base is a FromSoftware open-world action RPG?
Natural-language answer:
The game in the local knowledge base that is a FromSoftware open-world action RPG is **Elden Ring**.

Structured JSON:
{
  "question": "Which game in the local knowledge base is a FromSoftware open-world action RPG?",
  "answer": "The game in the local knowledge base that is a FromSoftware open-world action RPG is **Elden Ring**.",
  "sources": [
    "https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved"
  ],
  "used_web_search": false,
  "retrieval_useful": true,
  "memories_recalled": 3,
  "documents_retrieved": 5,
  "workflow": [
    "__entry__",
    "recall_game_memory",
    "retrieve_game",
    "evaluate_retrieval",
    "synthesize_answer",
    "store_game_memory"
  ],
  "confidence": 0.9
}

Final-state evidence summary:
{
  "retrieved_games": [
    "Elden Ring",
    "Marvel's Spider-Man",
    "Baldur's Gate 3",
    "Hollow Knight",
    "The Legend of Ze


Question: Was Mortal Kombat X released natively for PlayStation 5?
Natural-language answer:
Mortal Kombat X was not released natively for PlayStation 5; it was originally released for PlayStation 4. However, it is playable on PlayStation 5, though some features may be absent. [Source](https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved).

Structured JSON:
{
  "question": "Was Mortal Kombat X released natively for PlayStation 5?",
  "answer": "Mortal Kombat X was not released natively for PlayStation 5; it was originally released for PlayStation 4. However, it is playable on PlayStation 5, though some features may be absent. [Source](https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved).",
  "sources": [
    "https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved",
    "https://www.facebook.com/groups/374725147397441/posts/1245046263698654",
    "https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85L

In [20]:
# TEST 3: Dashboard with a real retrieval run
if structured_test_results:
    dashboard_result = structured_test_results[-1]
else:
    dashboard_result = research_game_with_structured_output(
        "Was Mortal Kombat X released natively for PlayStation 5?"
    )

dashboard_state = dashboard_result["final_state"]
dashboard_run = dashboard_result["run"]

print("Dashboard source run:")
print(json.dumps({
    "question": dashboard_result["structured_json"]["question"],
    "answer": dashboard_result["natural_language"],
    "workflow": dashboard_result["structured_json"]["workflow"],
    "retrieved_games": [
        item.get("Name") for item in dashboard_state.get("retrieved_docs", [])
    ],
    "web_sources": [
        item.get("url") for item in dashboard_state.get("web_results", [])
    ],
    "persistent_memory_count": persistent_memory_collection.count(),
}, indent=2, ensure_ascii=False, default=str))

render_research_dashboard(run=dashboard_run, final_state=dashboard_state)


Dashboard source run:
{
  "question": "Was Mortal Kombat X released natively for PlayStation 5?",
  "answer": "Mortal Kombat X was not released natively for PlayStation 5; it was originally released for PlayStation 4. However, it is playable on PlayStation 5, though some features may be absent. [Source](https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved).",
  "workflow": [
    "__entry__",
    "recall_game_memory",
    "retrieve_game",
    "evaluate_retrieval",
    "game_web_search",
    "synthesize_answer",
    "store_game_memory"
  ],
  "retrieved_games": [
    "God of War Ragnarok",
    "Marvel's Spider-Man 2",
    "Halo Infinite",
    "Gran Turismo 5",
    "Marvel's Spider-Man"
  ],
  "web_sources": [
    "https://www.facebook.com/groups/374725147397441/posts/1245046263698654",
    "https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V",
    "https://en.wikipedia.org/wiki/Mortal_Kombat_X",
    "https://www.di

In [21]:
# TEST 4: Custom tools directly and through the custom-tools agent
review_examples = [
    "An amazing, polished game with beautiful art and smooth combat.",
    "A buggy, repetitive and disappointing release that feels unfinished.",
    "The game has a large map and several playable characters.",
]

print("Direct sentiment-tool tests:")
for review in review_examples:
    result = analyze_game_review_sentiment(review)
    print(f"\nReview: {review}")
    print(json.dumps(result["output"], indent=2, ensure_ascii=False))

custom_agent_prompts = [
    "Call analyze_game_review_sentiment for this review and explain the result: "
    "An amazing and immersive game, but some missions are repetitive.",
    "Call detect_trending_games to identify currently trending Nintendo Switch games. "
    "Summarize the evidence and cite the result URLs.",
]

custom_tools_agent.reset_session()
custom_agent_runs = []
for index, prompt in enumerate(custom_agent_prompts, start=1):
    with redirect_stdout(io.StringIO()):
        run = custom_tools_agent.invoke(
            prompt,
            session_id=f"custom-tools-test-{index}",
        )
    custom_agent_runs.append(run)

for index, (prompt, run) in enumerate(zip(custom_agent_prompts, custom_agent_runs), start=1):
    print(f"\n{'=' * 96}")
    print(f"Custom-tools agent test {index}")
    print(f"Prompt: {prompt}")
    print(f"Workflow snapshots: {' -> '.join(snapshot.step_id for snapshot in run.snapshots)}")
    messages = (run.get_final_state() or {}).get("messages", [])
    for message in messages:
        for call in getattr(message, "tool_calls", None) or []:
            print(f"\nTool called: {call.function.name}")
            print(f"Arguments: {call.function.arguments}")
        if message.role == "tool":
            payload = decode_tool_message(message.content)
            print(f"Tool result ({message.name}):")
            print(json.dumps(compact_tool_output(message.name, payload), indent=2, ensure_ascii=False, default=str))

    final_answer = next(
        (
            message.content
            for message in reversed(messages)
            if message.role == "assistant" and message.content
        ),
        "No final answer produced.",
    )
    print("\nFinal agent answer:")
    print(final_answer)


Direct sentiment-tool tests:

Review: An amazing, polished game with beautiful art and smooth combat.
{
  "sentiment": "positive",
  "score": 4,
  "confidence": 1.0,
  "positive_terms": [
    "amazing",
    "polished",
    "beautiful",
    "smooth"
  ],
  "negative_terms": []
}

Review: A buggy, repetitive and disappointing release that feels unfinished.
{
  "sentiment": "negative",
  "score": -4,
  "confidence": 1.0,
  "positive_terms": [],
  "negative_terms": [
    "buggy",
    "repetitive",
    "disappointing",
    "unfinished"
  ]
}

Review: The game has a large map and several playable characters.
{
  "sentiment": "neutral",
  "score": 0,
  "confidence": 0.0,
  "positive_terms": [],
  "negative_terms": []
}



Custom-tools agent test 1
Prompt: Call analyze_game_review_sentiment for this review and explain the result: An amazing and immersive game, but some missions are repetitive.
Workflow snapshots: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor

Tool called: analyze_game_review_sentiment
Arguments: {"review":"An amazing and immersive game, but some missions are repetitive."}
Tool result (analyze_game_review_sentiment):
{
  "sentiment": "positive",
  "score": 1,
  "confidence": 0.33,
  "positive_terms": [
    "amazing",
    "immersive"
  ],
  "negative_terms": [
    "repetitive"
  ]
}

Final agent answer:
The analysis of the review indicates a **positive sentiment** overall, with a sentiment score of **1**. However, the confidence level in this sentiment is relatively low at **0.33**. 

### Breakdown of the Results:
- **Positive Terms**: The words "amazing" and "immersive" contribute to the positive sentiment, highlighting the reviewer's appreciation for the g